# Day 8 — RAG Pipeline v2 · Demo

Notebook trình bày end-to-end pipeline RAG về **pháp luật ma tuý** + **tin tức nghệ sĩ liên quan ma tuý**.

**Nội dung:**
1. Dữ liệu (Task 1–3)
2. Chunking & Indexing (Task 4)
3. Semantic / Lexical / Reranking (Task 5–7)
4. PageIndex fallback (Task 8)
5. Retrieval pipeline + Generation có citation (Task 9–10)
6. **Bonus:** HyDE
7. **Bonus:** Conversation memory
8. **Bonus:** Lexical search khác BM25 (giải thích)
9. Evaluation (golden dataset + A/B)

In [ ]:
# Setup: thêm project root vào sys.path để import package src
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

## 1. Dữ liệu (Task 1–3)
Văn bản luật (PDF/DOCX) + bài báo (JSON) → MarkItDown → markdown chuẩn hoá trong `data/standardized/`.

In [ ]:
from src.rag_utils import load_markdown_documents

docs = load_markdown_documents()
print(f'Tổng số tài liệu chuẩn hoá: {len(docs)}')
for d in docs:
    print(f"  - [{d['metadata']['type']:5}] {d['metadata']['source']:35} ({len(d['content'])} ký tự)")

## 2. Chunking & Indexing (Task 4)
Cắt theo đoạn văn, `chunk_size=500`, `chunk_overlap=50` (cân bằng giữa đủ ngữ cảnh và tránh lost-in-the-middle).

In [ ]:
from src.rag_utils import default_chunks

chunks = default_chunks(chunk_size=500, chunk_overlap=50)
print(f'Tổng số chunks: {len(chunks)}')
print('Ví dụ chunk[0]:')
print(chunks[0]['content'][:300], '...')

## 3. Semantic / Lexical / Reranking (Task 5–7)

In [ ]:
from src.task5_semantic_search import semantic_search
from src.task6_lexical_search import lexical_search
from src.task7_reranking import rerank

query = 'Quy trình cai nghiện ma tuý gồm những giai đoạn nào?'

print('— SEMANTIC (Task 5) —')
for r in semantic_search(query, top_k=3):
    print(f"  [{r['score']:.3f}] {r['metadata']['source']} | {r['content'][:60]}...")

print('\n— LEXICAL / BM25 (Task 6) —')
for r in lexical_search(query, top_k=3):
    print(f"  [{r['score']:.3f}] {r['metadata']['source']} | {r['content'][:60]}...")

print('\n— RERANK (Task 7) —')
cands = semantic_search(query, top_k=6)
for r in rerank(query, cands, top_k=3):
    print(f"  [{r['score']:.3f}] {r['metadata']['source']} | {r['content'][:60]}...")

## 4–5. Retrieval Pipeline (Task 9) + Generation có citation (Task 10)
`retrieve()` = semantic + lexical → RRF merge → rerank → (fallback PageIndex nếu score thấp).

In [ ]:
from src.task10_generation import generate_with_citation

result = generate_with_citation('Hình phạt cho tội tàng trữ trái phép chất ma tuý?', top_k=3)
print('ANSWER:\n', result['answer'])
print('\nSOURCES:')
for s in result['sources']:
    print(f"  - {s['metadata']['source']} (score {s.get('score', 0):.3f})")

## 6. Bonus — HyDE (Hypothetical Document Embeddings)
Sinh một *tài liệu giả định* cho câu hỏi rồi đối sánh tài liệu đó với corpus → thu hẹp khoảng cách lexical, tăng recall cho câu hỏi diễn đạt khác từ khoá.
(Dùng LLM nếu có API key, ngược lại mở rộng từ khoá cùng miền.)

In [ ]:
from src.task_bonus_hyde import generate_hypothetical_document, hyde_search
from src.task5_semantic_search import semantic_search

q = 'Người nổi tiếng nào dính líu chất cấm?'
print('Pseudo-document:\n ', generate_hypothetical_document(q)[:200], '...')

print('\n— Semantic thường (top 3) —')
for r in semantic_search(q, top_k=3):
    print(f"  [{r['score']:.3f}] {r['metadata']['source']}")

print('\n— HyDE (top 3) —')
for r in hyde_search(q, top_k=3):
    print(f"  [{r['score']:.3f}] {r['metadata']['source']}")

## 7. Bonus — Conversation Memory (multi-turn)
Viết lại câu hỏi follow-up thành câu hỏi độc lập dựa trên lịch sử phiên.

In [ ]:
from src.conversation import add_turn, condense_query, reset_session

sid = 'demo-notebook'
reset_session(sid)
add_turn(sid, 'user', 'DJ Thái Hoàng bị bắt vì lý do gì?')
add_turn(sid, 'assistant', 'DJ Thái Hoàng bị bắt quả tang vì tàng trữ trái phép chất ma tuý.')

follow = 'Còn anh ấy bị xử lý ở đâu?'
print('Follow-up gốc :', follow)
print('Đã viết lại   :', condense_query(sid, follow))

## 8. Bonus — Lexical search khác BM25 (giải thích cơ chế)

Module Task 6 mặc định dùng **BM25** (rank-bm25). Một phương án lexical thay thế là **TF-IDF + cosine similarity**:

- **TF-IDF**: mỗi token có trọng số `tf * idf`, trong đó `idf = log(N / df)` — token hiếm (xuất hiện ở ít văn bản) được trọng số cao. Tài liệu và truy vấn được biểu diễn thành vector trong không gian từ vựng, độ liên quan = **cosine similarity** giữa hai vector.
- **Khác biệt chính so với BM25**:
  - BM25 có **bão hoà tần suất** (term frequency saturation, tham số `k1`) — lặp từ nhiều lần không tăng điểm tuyến tính; TF-IDF thuần thì tăng tuyến tính theo `tf`.
  - BM25 **chuẩn hoá theo độ dài tài liệu** (`b`) để tài liệu dài không bị thiên vị; TF-IDF chỉ chuẩn hoá qua chuẩn vector (cosine).
  - ⇒ BM25 thường tốt hơn trên văn bản dài, độ dài không đồng đều (đúng với văn bản luật ở đây).

Cell dưới minh hoạ TF-IDF như một lexical scorer thay thế.

In [ ]:
import math
from collections import Counter
from src.rag_utils import ensure_chunks, tokenize, cosine_from_counters

def tfidf_search(query, top_k=3):
    chunks = ensure_chunks()
    docs_tokens = [tokenize(c['content']) for c in chunks]
    N = len(docs_tokens)
    df = Counter()
    for toks in docs_tokens:
        for t in set(toks):
            df[t] += 1
    idf = {t: math.log(N / (1 + d)) + 1 for t, d in df.items()}
    def vec(tokens):
        tf = Counter(tokens)
        return Counter({t: tf[t] * idf.get(t, 0.0) for t in tf})
    qv = vec(tokenize(query))
    scored = [(cosine_from_counters(qv, vec(toks)), chunks[i]) for i, toks in enumerate(docs_tokens)]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

for score, ch in tfidf_search('cai nghiện ma tuý bắt buộc', top_k=3):
    print(f"  [{score:.3f}] {ch['metadata']['source']} | {ch['content'][:60]}...")

## 9. Evaluation — Golden dataset + A/B
Chạy `eval_pipeline.py` đánh giá 4 metrics (Faithfulness, Answer Relevance, Context Recall, Context Precision) và so sánh A/B.

In [ ]:
from group_project.evaluation.eval_pipeline import load_golden_dataset, compare_configs

golden = load_golden_dataset()
print(f'Golden dataset: {len(golden)} cặp Q&A\n')
results = compare_configs(golden)
for name, res in results.items():
    print(f"\n{name} ({res['label']}):")
    for k, v in res['agg'].items():
        print(f'  {k:20}: {v}')

---
**Kết quả chi tiết:** xem `group_project/evaluation/results.md`.

**Chạy chatbot demo:** `python chat_server.py` → mở http://127.0.0.1:8008